## 0. Configurando sessão spark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-628efeff-1178-416b-aa1b-2a7d98e39663;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 167ms :: artifacts dl 4ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [4]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [5]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_municipio = f"{par_source_project}.silver.municipio"
par_source_silver_uf = f"{par_source_project}.silver.uf"
par_source_silver_aluno = f"{par_source_project}.silver.aluno"

par_source_gold_rede = f"{par_source_project}.gold.dim_rede"

## 3. Leitura dos dados da origem

In [6]:
df_scr_municipio = spark.read.format("bigquery").option("table",par_source_silver_municipio).load()
df_scr_uf = spark.read.format("bigquery").option("table",par_source_silver_uf).load()
df_scr_aluno = spark.read.format("bigquery").option("table",par_source_silver_aluno).load()

## 4. Transformações

### 4.1. rede de todas as tabelas

In [9]:
dim_rede = (
    df_scr_municipio.select("rede_id","rede")
    .union(df_scr_aluno.select("rede_id","rede"))
    .union(df_scr_uf.select("rede_id","rede"))
    .distinct()
    .filter(F.col("rede_id").isNotNull())
)

## 5. Armazenamento no BQ

In [11]:
(
    dim_rede.write.format("bigquery")
    .option("table", par_source_gold_rede)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)